# NumPy: Broadcasting & Vectorization

**Interview focus:** broadcasting rules, avoiding Python loops, ufuncs, and performance.

In [ ]:
import numpy as np

## 1. Broadcasting Rules

Two arrays are broadcastable if, for each dimension (from the trailing end):
1. Dimensions are equal, OR
2. One of them is 1, OR
3. One of them doesn't exist (missing dim is treated as 1)

In [ ]:
# Scalar broadcast
a = np.array([1, 2, 3])
print(a + 10)  # [11, 12, 13]

# 1D + 2D
b = np.arange(12).reshape(3, 4)
c = np.array([1, 2, 3, 4])
print((b + c).shape)  # (3, 4) — c broadcast along rows

# Column vector + row vector → outer sum
col = np.array([[1], [2], [3]])   # (3, 1)
row = np.array([10, 20, 30, 40]) # (4,)
outer_sum = col + row              # (3, 4)
print(outer_sum)

In [ ]:
# Broadcasting failure example — uncomment to see error
# a = np.ones((3, 4))
# b = np.ones((2, 4))
# a + b  # ValueError: operands could not be broadcast

# Fix with explicit reshape
a = np.ones((3, 1))
b = np.ones((1, 4))
print((a + b).shape)  # (3, 4)

## 2. Universal Functions (ufuncs)

In [ ]:
x = np.array([1, 4, 9, 16], dtype=float)

# Element-wise
print(np.sqrt(x))
print(np.exp(x))
print(np.log(x + 1))

# Comparison ufuncs return boolean arrays
print(x > 5)
print(np.greater(x, 5))

# Aggregations
data = np.random.randn(1000, 5)
print(f"mean per col: {data.mean(axis=0).shape}")
print(f"sum per row:  {data.sum(axis=1).shape}")

## 3. Vectorization vs Loops

In [ ]:
import time

n = 1_000_000
a = np.random.randn(n)
b = np.random.randn(n)

# Python loop
start = time.time()
result_loop = [a[i] * b[i] + 1 for i in range(n)]
t_loop = time.time() - start

# Vectorized
start = time.time()
result_vec = a * b + 1
t_vec = time.time() - start

print(f"Loop:        {t_loop:.4f}s")
print(f"Vectorized:  {t_vec:.4f}s")
print(f"Speedup:     {t_loop / t_vec:.1f}x")

## 4. Common Interview Patterns

In [ ]:
# Softmax (numerically stable)
def softmax(x):
  x = x - x.max(axis=-1, keepdims=True)  # prevent overflow
  exp_x = np.exp(x)
  return exp_x / exp_x.sum(axis=-1, keepdims=True)

logits = np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]])
print(softmax(logits))
print(softmax(logits).sum(axis=1))  # should be [1., 1.]

In [ ]:
# Pairwise distances without loops
# Given X (n, d), compute (n, n) distance matrix
X = np.random.randn(5, 3)

# ||x_i - x_j||^2 = ||x_i||^2 + ||x_j||^2 - 2 * x_i . x_j
sq_norms = (X ** 2).sum(axis=1, keepdims=True)  # (n, 1)
dist_sq = sq_norms + sq_norms.T - 2 * X @ X.T
dist = np.sqrt(np.maximum(dist_sq, 0))  # clip negatives from float error
print(dist.round(2))

In [ ]:
# One-hot encoding
def one_hot(labels, num_classes):
  return np.eye(num_classes)[labels]

labels = np.array([0, 2, 1, 0])
print(one_hot(labels, 3))

## 5. Practice Problems

In [ ]:
# Problem: Normalize each row to sum to 1 (like converting counts to probabilities)
counts = np.array([[1, 2, 3], [4, 0, 6], [0, 0, 0]])
row_sums = counts.sum(axis=1, keepdims=True)
probs = np.where(row_sums > 0, counts / row_sums, 0)
print(probs)

# Problem: Implement ReLU and its derivative
def relu(x):
  return np.maximum(0, x)

def relu_grad(x):
  return (x > 0).astype(float)

x = np.array([-2, -1, 0, 1, 2], dtype=float)
print(relu(x), relu_grad(x))